In [9]:
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# 1) Create sample data
rng = np.random.default_rng(42)
n = 200
X = rng.normal(size=(n, 4))
y = 4 * X[:, 0] - 2 * X[:, 1] + 0.8 * X[:, 2] + rng.normal(scale=1.0, size=n)

# 2) Candidate hyperparameters to test
alphas = [0.001, 0.01, 0.1, 1, 10, 100]

# 3) Cross-validation setup
cv = KFold(n_splits=5, shuffle=True, random_state=42)

# 4) Evaluate each alpha with CV
rows = []
for alpha in alphas:
    model = Pipeline([("scale", StandardScaler()), ("ridge", Ridge(alpha=alpha))])
    # negative RMSE -> convert to positive RMSE
    cv_scores = cross_val_score(model, X, y, cv=cv, scoring="neg_root_mean_squared_error")
    rows.append({"alpha": alpha, "mean_rmse": -cv_scores.mean(), "std_rmse": cv_scores.std()})

cv_results = pd.DataFrame(rows).sort_values("mean_rmse", ascending=True)
best_alpha = cv_results.iloc[0]["alpha"]

# 5) Refit best model on all data
best_model = Pipeline([("scale", StandardScaler()), ("ridge", Ridge(alpha=best_alpha))]).fit(X, y)


# Useful outputs
print("Cross-validation results")
print(cv_results)
print(
    {
        "best_alpha": best_alpha,
        "best_coefficients": best_model.named_steps["ridge"].coef_,
        "best_intercept": best_model.named_steps["ridge"].intercept_,
    }
)

# 6) Use the best model to make predictions
predictions = best_model.predict(X)

# Print the predictions and actual value
print("Predictions vs Actual")
for pred, actual in zip(predictions[:10], y[:10]):
    print(f"Predicted: {pred:.3f}, Actual: {actual:.3f}")

Cross-validation results
     alpha  mean_rmse  std_rmse
2    0.100   1.020228  0.163627
1    0.010   1.020253  0.163691
0    0.001   1.020256  0.163698
3    1.000   1.020395  0.162962
4   10.000   1.057752  0.153667
5  100.000   2.079491  0.142524
{'best_alpha': np.float64(0.1), 'best_coefficients': array([ 4.02655017, -2.02126276,  0.84977185, -0.04558906]), 'best_intercept': np.float64(-0.21015763598215614)}
Predictions vs Actual
Predicted: 3.903, Actual: 3.054
Predicted: -5.144, Actual: -6.181
Predicted: 2.341, Actual: 2.687
Predicted: -1.620, Actual: -1.237
Predicted: 4.154, Actual: 5.383
Predicted: 1.646, Actual: 2.700
Predicted: -0.614, Actual: -0.715
Predicted: 2.588, Actual: 1.258
Predicted: 0.019, Actual: -0.248
Predicted: 0.490, Actual: 0.782
